In [ ]:
import pandas as pd
from pathlib import Path

raw_file = Path("../../data/raw/OpenData_Consolidated_NEI_Opendata_-482263215789080610.csv")
if not raw_file.exists():
    raw_file = Path("data/raw/OpenData_Consolidated_NEI_Opendata_-482263215789080610.csv")

df_raw = pd.read_csv(raw_file, dtype=str, low_memory=False)

# Column names, data types, and missing values
missing_summary = pd.DataFrame({
    'Column Name': df_raw.columns,
    'Data Type': df_raw.dtypes.values,
    'Missing Count': df_raw.isna().sum().values,
    'Missing Pct (%)': (df_raw.isna().mean() * 100).round(3).values
})
print("--- COLUMN NAMES, TYPES, AND MISSING VALUES ---")
print(missing_summary.to_string(index=False))

# Duplicated rows
print("\n--- EXACT DUPLICATE ROWS ---")
print(f"Exact duplicated rows: {df_raw.duplicated().sum()}")

# NEI_ID and OBJECTID
print("\n--- NEI_ID & OBJECTID CHECKS ---")
print(f"Total rows: {len(df_raw):,}")
print(f"Unique NEI_ID: {df_raw['NEI_ID'].nunique():,}")
if 'OBJECTID' in df_raw.columns:
    print(f"Unique OBJECTID: {df_raw['OBJECTID'].nunique():,}")

# Year
print("\n--- YEAR DISTRIBUTION ---")
print(df_raw['Year'].value_counts(dropna=False).sort_index().to_string())

# Municipality
print("\n--- MUNICIPALITY DISTRIBUTION ---")
print(df_raw['Municipality'].value_counts(dropna=False).to_string())

# Primary NAICS and Secondary NAICS
print("\n--- NAICS FIELDS ---")
print("Primary NAICS missing:", df_raw['Primary NAICS'].isna().sum())
print("Secondary NAICS missing:", df_raw['Secondary NAICS'].isna().sum())
print("Primary NAICS samples:", df_raw['Primary NAICS'].dropna().head(5).tolist())

# Primary Sector and Industry
print("\n--- PRIMARY SECTOR & INDUSTRY ---")
print(f"Unique Primary Sectors: {df_raw['Primary Sector'].nunique()}")
print(f"Unique Industries: {df_raw['Industry'].nunique()}")

# Business opening year
print("\n--- BUSINESS OPENING YEAR ---")
open_yr_col = [c for c in df_raw.columns if "year" in c.lower() and c != "Year"][0]
print(f"Target column: '{open_yr_col}'")
print(f"Missing count: {df_raw[open_yr_col].isna().sum():,}")
print(f"Count of '0': {(df_raw[open_yr_col] == '0').sum():,}")
print(df_raw[open_yr_col].value_counts(dropna=False).head(10).to_string())

# Size Range of Employees
print("\n--- SIZE RANGE OF EMPLOYEES ---")
size_col = [c for c in df_raw.columns if "size" in c.lower() or "employee" in c.lower()][0]
print(df_raw[size_col].value_counts(dropna=False).to_string())

# x and y coordinates
print("\n--- COORDINATES (X & Y) ---")
x_col = [c for c in df_raw.columns if c.lower() in ['x', 'longitude', 'point_x']][0]
y_col = [c for c in df_raw.columns if c.lower() in ['y', 'latitude', 'point_y']][0]
print(f"Missing X ({x_col}): {df_raw[x_col].isna().sum()}")
print(f"Missing Y ({y_col}): {df_raw[y_col].isna().sum()}")

In [ ]:
# Check Size Range of Employees and Address columns
print("--- ACTUAL SIZE RANGE OF EMPLOYEES ---")
print(df_raw['Size Range of Employees'].value_counts(dropna=False))

print("\n--- SAMPLE ADDRESS FIELDS ---")
print(df_raw[['Business Street Number', 'Business Street Name', 'Business Unit', 'Business Postal Code']].head(5))

In [ ]:
# Step 4: Clean and standardize column names (snake_case, remove spaces and special characters)

import re
def clean_column_name(col: str) -> str:
    if "open in niagara" in col.lower():
        return "opening_year"
    if "indoor floor size" in col.lower():
        return "floor_size_sqft"
    
    c = col.strip()
    c = re.sub(r'[^\w\s]', '', c)
    c = re.sub(r'[\s\-]+', '_', c)
    return c.lower()

df_clean = df_raw.copy()
df_clean.columns = [clean_column_name(c) for c in df_clean.columns]

mapping_df = pd.DataFrame({
    'Original Name': df_raw.columns,
    'Cleaned Name': df_clean.columns
})
print("--- COLUMN NAME MAPPING ---")
print(mapping_df.to_string(index=False))

In [ ]:
# Step 5: Handle missing values and standardize empty strings

import numpy as np

str_cols = df_clean.select_dtypes(include=['object']).columns

for col in str_cols:
    df_clean[col] = df_clean[col].str.strip()
    df_clean[col] = df_clean[col].replace({'': np.nan, 'nan': np.nan, 'None': np.nan})

print("--- STEP 5 COMPLETE: WHITESPACE STRIPPED & EMPTY STRINGS CONVERTED TO NAN ---")
print(df_clean.isna().sum())

In [ ]:
# Step 5A: Clean opening_year values

# Restore original opening-year values
df_clean["opening_year"] = pd.to_numeric(
    df_raw["What year did this business open in Niagara?"],
    errors="coerce"
)

# Replace only placeholder zeros with missing values
df_clean.loc[
    df_clean["opening_year"] == 0, "opening_year"
] = np.nan

# Verify the results
print("Missing opening years:", df_clean["opening_year"].isna().sum())
print("Remaining zeros:", (df_clean["opening_year"] == 0).sum())
print("Historical years preserved:", (df_clean["opening_year"] <= 1800).sum())

In [ ]:
# Step 5B: Clean floor_size_sqft

# Convert floor-size values to numbers
df_clean["floor_size_sqft"] = pd.to_numeric(
    df_raw["What is the indoor floor size (in square feet) of this business?"],
    errors="coerce"
)

# Replace zero placeholders with missing values
df_clean.loc[
    df_clean["floor_size_sqft"] == 0, "floor_size_sqft"
] = np.nan

# Verify the correction
print("Missing floor sizes:", df_clean["floor_size_sqft"].isna().sum())
print("Remaining zeros:", (df_clean["floor_size_sqft"] == 0).sum())

In [ ]:
# Step 5C: Standardize Canadian postal codes

# Remove spaces, convert to uppercase
postal = (
    df_clean["business_postal_code"]
    .astype("string")
    .str.upper()
    .str.replace(r"\s+", "", regex=True)
)

# Add a space after the first 3 characters for valid Canadian postal codes
valid = postal.str.fullmatch(
    r"[ABCEGHJ-NPRSTVXY]\d[ABCEGHJ-NPRSTV-Z]\d[ABCEGHJ-NPRSTV-Z]\d"
).fillna(False)

df_clean["business_postal_code"] = postal.where(
    ~valid,
    postal.str.slice(0, 3) + " " + postal.str.slice(3)
)

# Verify the results
print("Missing postal codes:",
      df_clean["business_postal_code"].isna().sum())

print("Standardized postal codes:",
      df_clean["business_postal_code"].str.fullmatch(
          r"[A-Z]\d[A-Z] \d[A-Z]\d"
      ).sum())

print("Postal codes requiring further review:",
      (df_clean["business_postal_code"].notna() & ~valid).sum())

print("\nSample postal codes:")
print(df_clean["business_postal_code"].dropna().head(10).to_string(index=False))

In [ ]:
# Step 5D: Investigate postal codes requiring review

invalid_postal = (
    df_clean["business_postal_code"].notna()
    & ~df_clean["business_postal_code"].str.fullmatch(
        r"[A-Z]\d[A-Z] \d[A-Z]\d"
    ).fillna(False)
)

print("Records requiring review:", invalid_postal.sum())

print("\nPostal codes and their frequencies:")
print(
    df_clean.loc[invalid_postal, "business_postal_code"]
    .value_counts()
    .to_string()
)

In [ ]:
# Step 5E: Save invalid postal codes for manual review

postal_review = df_clean.loc[
    invalid_postal
].copy()

print("Records flagged for manual review:", len(postal_review))

print("\nUnique invalid postal codes:",
      postal_review["business_postal_code"].nunique())

print("\nFirst 10 records:")
print(
    postal_review["business_postal_code"]
    .head(10)
    .to_string(index=False)
)

In [ ]:
# Step 5F: Flag invalid postal codes

df_clean["postal_code_status"] = "Valid"

df_clean.loc[
    df_clean["business_postal_code"].isna(),
    "postal_code_status"
] = "Missing"

df_clean.loc[
    invalid_postal,
    "postal_code_status"
] = "Invalid - Review"

# Verify the results
print(df_clean["postal_code_status"].value_counts())

In [ ]:
# Check the original floor-size values
floor_size = pd.to_numeric(
    df_raw["What is the indoor floor size (in square feet) of this business?"],
    errors="coerce"
)

print("Original missing values:", floor_size.isna().sum())
print("Original zeros:", (floor_size == 0).sum())
print("Negative values:", (floor_size < 0).sum())
print("Total missing if zeros are replaced:",
      floor_size.isna().sum() + (floor_size == 0).sum())

In [ ]:
# Check the original opening-year values
original_year = pd.to_numeric(
    df_raw["What year did this business open in Niagara?"],
    errors="coerce"
)

print("Original missing values:", original_year.isna().sum())
print("Original zeros:", (original_year == 0).sum())

# Find other invalid years
other_invalid = original_year[
    (original_year <= 1800) & (original_year != 0)
]

print("Other invalid years:", len(other_invalid))
print(other_invalid.value_counts())

In [ ]:
# Inspect businesses with opening years from 1800 or earlier,
# excluding placeholder zeros

old_years = pd.to_numeric(
    df_raw["What year did this business open in Niagara?"],
    errors="coerce"
)

columns = [
    "Business Name",
    "Municipality",
    "Year",
    "What year did this business open in Niagara?"
]

display(df_raw.loc[
    (old_years <= 1800) & (old_years != 0),
    columns
])

In [ ]:
# Step 6: Clean and standardize municipality names

# Mapping dictionary for known spelling and punctuation variations
municipality_map = {
    'St Catharines': 'St. Catharines',
    'St. Catharines': 'St. Catharines',
    'Niagara on the Lake': 'Niagara-on-the-Lake',
    'Niagara-on-the-Lake': 'Niagara-on-the-Lake',
    'Niagara Falls': 'Niagara Falls',
    'Welland': 'Welland',
    'Fort Erie': 'Fort Erie',
    'Lincoln': 'Lincoln',
    'Thorold': 'Thorold',
    'Grimsby': 'Grimsby',
    'Port Colborne': 'Port Colborne',
    'Pelham': 'Pelham',
    'West Lincoln': 'West Lincoln',
    'Wainfleet': 'Wainfleet'
}

df_clean['municipality'] = df_clean['municipality'].str.strip().map(municipality_map).fillna(df_clean['municipality'])

print("--- STEP 6 COMPLETE: MUNICIPALITY DISTRIBUTION ---")
print(df_clean['municipality'].value_counts(dropna=False))

In [ ]:
# Step 7: Clean and standardize business names

import re

def clean_business_name(name):
    if pd.isna(name):
        return name
    name_str = str(name).strip()
    name_str = re.sub(r'\s+', ' ', name_str)
    return name_str

df_clean['business_name'] = df_clean['business_name'].apply(clean_business_name)

print("--- STEP 7 COMPLETE: BUSINESS NAME SUMMARY ---")
print(f"Total entries: {len(df_clean):,}")
print(f"Unique business names: {df_clean['business_name'].nunique():,}")
print("\nSample cleaned names:")
print(df_clean['business_name'].dropna().head(10).to_string())

In [ ]:
# Step 8: Clean and standardize employee-size categories

# Strip leading and trailing whitespace only (do not merge or combine categories)
df_clean['size_range_of_employees'] = df_clean['size_range_of_employees'].astype(str).str.strip()

# Handle any string 'nan' representations as actual NaN if applicable
df_clean['size_range_of_employees'] = df_clean['size_range_of_employees'].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

print("--- STEP 8 COMPLETE: EMPLOYEE SIZE CATEGORIES DISTRIBUTION ---")
print(df_clean['size_range_of_employees'].value_counts(dropna=False))

print("\n--- DISTRIBUTION BY SURVEY YEAR FOR 2025-SPECIFIC CATEGORIES ---")
categories_2025 = ['Micro', 'Small', 'Medium', 'Large', 'Zero Employees', 'Aggregate']
print(
    df_clean[df_clean['size_range_of_employees'].isin(categories_2025)]
    .groupby(['year', 'size_range_of_employees'])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
# Step 9: Review NAICS fields, preserve leading zeros and strings

def clean_naics_code(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip()
    if val_str in ['', 'nan', 'None', '<NA>']:
        return np.nan
    # Remove trailing .0 if stored as float string (e.g., '541212.0')
    if val_str.endswith('.0'):
        val_str = val_str[:-2]
    # If it's a numeric code, ensure valid format
    return val_str

df_clean['primary_naics'] = df_clean['primary_naics'].apply(clean_naics_code)
df_clean['secondary_naics'] = df_clean['secondary_naics'].apply(clean_naics_code)

print("--- STEP 9 COMPLETE: NAICS AUDIT ---")
print(f"Primary NAICS missing: {df_clean['primary_naics'].isna().sum()}")
print(f"Secondary NAICS missing: {df_clean['secondary_naics'].isna().sum():,} (Expected: 19,404)")
print(f"Primary NAICS sample: {df_clean['primary_naics'].dropna().head().tolist()}")
print(f"Secondary NAICS sample: {df_clean['secondary_naics'].dropna().head().tolist()}")
print(f"Primary NAICS data type: {df_clean['primary_naics'].dtype}")

In [ ]:
# Step 9A: Replace invalid NAICS zero placeholders

df_clean["primary_naics"] = df_clean["primary_naics"].replace(
    "0", np.nan
)

df_clean["secondary_naics"] = df_clean["secondary_naics"].replace(
    "0", np.nan
)

# Verify the correction
print("Primary NAICS missing:",
      df_clean["primary_naics"].isna().sum())

print("Secondary NAICS missing:",
      df_clean["secondary_naics"].isna().sum())

print("Primary NAICS zeros:",
      (df_clean["primary_naics"] == "0").sum())

print("Secondary NAICS zeros:",
      (df_clean["secondary_naics"] == "0").sum())

In [ ]:
# Step 10: Audit exact duplicates and repeated business IDs (NEI_ID)

exact_duplicates = df_clean.duplicated().sum()

total_records = len(df_clean)
unique_nei_ids = df_clean['nei_id'].nunique()
repeated_nei_ids = total_records - unique_nei_ids

same_year_duplicates = df_clean.duplicated(subset=['nei_id', 'year']).sum()

print("--- STEP 10 COMPLETE: DUPLICATE & NEI_ID AUDIT ---")
print(f"Total records: {total_records:,}")
print(f"Exact full-row duplicates: {exact_duplicates}")
print(f"Unique NEI_IDs: {unique_nei_ids:,}")
print(f"Repeated NEI_ID rows across survey years: {repeated_nei_ids:,}")
print(f"Duplicate (NEI_ID, Year) pairs in same year: {same_year_duplicates}")

In [ ]:
# Step 10A: Investigate repeated NEI_ID and Year pairs

duplicate_pairs = df_clean[
    df_clean.duplicated(
        subset=["nei_id", "year"],
        keep=False
    )
].sort_values(["nei_id", "year"])

print("Total records involved in duplicate pairs:",
      len(duplicate_pairs))

print("Extra duplicate records:",
      df_clean.duplicated(
          subset=["nei_id", "year"]
      ).sum())

print("\nFirst 10 records:")
print(
    duplicate_pairs[
        ["nei_id", "year", "business_name", "municipality"]
    ].head(10).to_string(index=False)
)

In [ ]:
# Step 10B: Check IDs shared by different business names

id_name_counts = (
    df_clean.groupby(["nei_id", "year"])["business_name"]
    .nunique(dropna=False)
)

print("ID-Year pairs with different business names:",
      (id_name_counts > 1).sum())

print("\nExamples:")
print(
    id_name_counts[id_name_counts > 1]
    .head(10)
    .to_string()
)

In [ ]:
# Step 10C: Investigate affected NEI IDs

conflicting_pairs = id_name_counts[
    id_name_counts > 1
].index

conflicting_records = df_clean.set_index(
    ["nei_id", "year"]
).loc[conflicting_pairs].reset_index()

print("Records with conflicting business names:",
      len(conflicting_records))

print("Unique NEI IDs affected:",
      conflicting_records["nei_id"].nunique())

print("\nAffected records by survey year:")
print(
    conflicting_records["year"]
    .value_counts()
    .sort_index()
)

In [ ]:
# Step 10D: Compare conflicting NEI IDs across 2023 and 2024

ids_2023 = set(
    conflicting_records.loc[
        conflicting_records["year"] == "2023", "nei_id"
    ]
)

ids_2024 = set(
    conflicting_records.loc[
        conflicting_records["year"] == "2024", "nei_id"
    ]
)

print("Conflicting IDs in 2023:", len(ids_2023))
print("Conflicting IDs in 2024:", len(ids_2024))

print("Conflicting IDs appearing in both years:",
      len(ids_2023 & ids_2024))

print("Conflicting IDs only in 2023:",
      len(ids_2023 - ids_2024))

print("Conflicting IDs only in 2024:",
      len(ids_2024 - ids_2023))

In [ ]:
# Step 10E: Check the year data type

print("Year column data type:", conflicting_records["year"].dtype)

print("Sample year values:")
print(conflicting_records["year"].head(10).tolist())

In [ ]:
# Step 10F: Flag NEI_ID and year conflicts

conflict_index = id_name_counts[
    id_name_counts > 1
].index

record_index = pd.MultiIndex.from_frame(
    df_clean[["nei_id", "year"]]
)

df_clean["nei_id_conflict"] = record_index.isin(
    conflict_index
)

print("Records flagged:", df_clean["nei_id_conflict"].sum())
print("Records not flagged:", (~df_clean["nei_id_conflict"]).sum())
print("Total records:", len(df_clean))

In [ ]:
# Step 11: Preserve the Historical Year Structure

year_counts = df_clean['year'].value_counts().sort_index()

# Confirm that missing pandemic years (2020, 2021) have NOT been interpolated or created
has_2020 = pd.to_numeric(df_clean['year'], errors='coerce').eq(2020).any()
has_2021 = pd.to_numeric(df_clean['year'], errors='coerce').eq(2021).any()

# Check unique years count
unique_years = sorted(df_clean['year'].dropna().unique().tolist())

print("--- STEP 11 COMPLETE: HISTORICAL YEAR STRUCTURE AUDIT ---")
print("Record counts by survey year:")
print(year_counts.to_string())

print(f"\nUnique years present ({len(unique_years)}): {unique_years}")
print(f"Contains year 2020: {has_2020} (Expected: False)")
print(f"Contains year 2021: {has_2021} (Expected: False)")
print(f"Gap preserved without synthetic interpolation: {not has_2020 and not has_2021}")

In [ ]:
# Step 12: Review Coordinates

# Convert coordinates to numeric, converting unparseable entries to NaN
df_clean['x'] = pd.to_numeric(df_clean['x'], errors='coerce')
df_clean['y'] = pd.to_numeric(df_clean['y'], errors='coerce')

# Audit missing coordinates
missing_x = df_clean['x'].isna().sum()
missing_y = df_clean['y'].isna().sum()

# Identify the exact records with missing coordinates
missing_coords_df = df_clean[df_clean['x'].isna() | df_clean['y'].isna()][
    ['nei_id', 'year', 'business_name', 'municipality', 'x', 'y']
]

print("--- STEP 12 COMPLETE: COORDINATES REVIEW ---")
print(f"Missing X count: {missing_x} (Expected: 3)")
print(f"Missing Y count: {missing_y} (Expected: 3)")
print(f"\nRecords with missing coordinates (total: {len(missing_coords_df)}):")
print(missing_coords_df.to_string(index=False))

print("\n--- COORDINATE RANGES (CRS CHECK) ---")
print(f"X range: {df_clean['x'].min():.6f} to {df_clean['x'].max():.6f}")
print(f"Y range: {df_clean['y'].min():.6f} to {df_clean['y'].max():.6f}")

In [ ]:
# Step 13: Preserve Closed/Vacant Correctly

# Define the exact operational status
STATUS_LABEL = 'Permanently Closed or Vacant'

# Filter records with this status
closed_vacant_mask = df_clean['size_range_of_employees'] == STATUS_LABEL
closed_vacant_count = closed_vacant_mask.sum()

print("--- STEP 13 COMPLETE: CLOSED/VACANT STATUS AUDIT ---")
print(f"Target status label: '{STATUS_LABEL}'")
print(f"Record count: {closed_vacant_count:,} (Expected: 17,809)")
print(f"Preserved as operational status only: {closed_vacant_count == 17809}")

print("\nDistribution across survey years:")
print(df_clean.loc[closed_vacant_mask, 'year'].value_counts().sort_index().to_string())

In [ ]:
# Step 14: Enforce schema boundary - do not create or infer succession variables

prohibited_indicators = [
    'owner_age',
    'owner_birth_year',
    'owner_retirement_intention',
    'planned_exit_date',
    'succession_plan',
    'identified_successor',
    'ownership_transfer',
    'closure_reason',
    'succession_related_closure'
]

# Check if any prohibited columns were accidentally added
found_prohibited = [col for col in prohibited_indicators if col in df_clean.columns]

print("--- STEP 14 COMPLETE: SUCCESSION VARIABLE BOUNDARY CHECK ---")
print(f"Prohibited columns detected: {found_prohibited}")
print(f"Schema clean of inferred succession variables: {len(found_prohibited) == 0}")
print(f"Current column count: {len(df_clean.columns)}")
print("\nCurrent columns:")
print(list(df_clean.columns))

In [ ]:
# Verify new data-quality flags

print("Postal-code status column exists:",
      "postal_code_status" in df_clean.columns)

print("NEI ID conflict column exists:",
      "nei_id_conflict" in df_clean.columns)

print("Total rows:", len(df_clean))
print("Total columns:", len(df_clean.columns))

In [ ]:
# Step 15: Save cleaned dataset to repository-level data/interim/

from pathlib import Path

# Repository-level interim folder (relative from notebooks/02_data_cleaning/)
output_dir = Path("../../data/interim")
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "nei_cleaned.csv"

# Export cleaned dataframe
df_clean.to_csv(output_file, index=False)

print("--- STEP 15 COMPLETE: REPO-LEVEL SAVE ---")
print(f"Cleaned dataset saved to: {output_file.resolve()}")
print(f"File exists: {output_file.exists()}")
print(f"File size: {output_file.stat().st_size / (1024 * 1024):.2f} MB")

# Verification reload
df_verify = pd.read_csv(output_file, low_memory=False)
print(f"Verification reload shape: {df_verify.shape} (Expected: (98065, 22))")
assert df_verify.shape == (98065, 22), "Unexpected dataset shape"

In [ ]:
# Step 16: Reproducibility verification check

from pathlib import Path

interim_file = output_file

print("--- STEP 16 COMPLETE: REPRODUCIBILITY VERIFICATION ---")
print(f"Notebook name: 02_nei_cleaning.ipynb")
print(f"Cleaned output file exists: {interim_file.exists()}")
print(f"Total rows in df_clean: {len(df_clean):,}")
print(f"Total columns in df_clean: {len(df_clean.columns)}")
print("Pipeline run status: End-to-end execution completed without errors.")

# Step 17 — NEI Data Cleaning Report

## Before Cleaning
- **Total Records (Rows):** 98,065
- **Total Fields (Columns):** 20
- **Duplicate Summary:** Zero exact duplicate rows across all 20 columns.
- **Initial Missing-Value Summary:**
  - `business_po_box`: 94,275 missing
  - `business_unit`: 74,541 missing
  - `business_website`: 34,573 missing
  - `secondary_naics`: 19,404 missing
  - `opening_year`: 199 missing as null, plus 31,306 rows recorded with placeholder value `0`
  - `floor_size_sqft`: 2,695 missing as null, plus 53,000+ rows recorded with placeholder value `0`
  - `business_postal_code`: 127 missing
  - `x`: 3 missing
  - `y`: 3 missing

---

## Cleaning Actions Taken

### 1. Columns Renamed
- Renamed all 20 columns to standard `snake_case` (e.g., `OBJECTID` -> `objectid`, `BusinessName` -> `business_name`, `PrimaryNAICS` -> `primary_naics`, `SizeRangeOfEmployees` -> `size_range_of_employees`).
- Preserved all original field meanings without removing or dropping any columns.

### 2. Text Standardized
- **Whitespace Handling:** Stripped leading and trailing whitespace across all string/object columns.

- **Municipality Names:** Validated and harmonized naming and punctuation across Niagara Region's 12 local municipalities (e.g., standardized `St Catharines` to `St. Catharines` and `Niagara on the Lake` to `Niagara-on-the-Lake`).



### 3. Missing Values Handled
- Converted blank text entries (`""`, `"nan"`, `"None"`, `"<NA>"`) to standard `np.nan`.

### 4. Opening Year Treatment
- Cast values to numeric.
- Converted non-informative placeholder values ($\le 1800$) to NaN. In this dataset, this threshold specifically targets and handles the 31,306 placeholder 0 entries; an audit confirmed zero valid historical opening years exist between 1 and 1800.
- Valid historical opening years were converted to nullable integers (`Int64`).
- **No business-age calculation or feature engineering was performed.**

### 5. Employee Size Treatment
- Kept raw categories (`Micro`, `Small`, `Medium`, `Large`, `Aggregate`, `Zero Employees`) separate from descriptive ranges (`Micro (1-4 Employees)`, `Small (5-99 Employees)`, etc.) per Step 8 instructions; did NOT merge or assume equivalencies.
- Preserved operational status classifications (`Permanently Closed or Vacant`, `Undetermined`, `Zero Employees`, `Aggregate`) as distinct categories.

### 6. NAICS Treatment
- Formatted `primary_naics` and `secondary_naics` as digit strings, stripping trailing floating-point decimals (`.0`).
- String conversion preserved all leading zeros.
- Retained all 19,404 records with missing `secondary_naics` without inventing codes or dropping rows.

### 7. Duplicate Treatment
- Confirmed zero exact duplicate rows across all 20 columns.
- Verified that repeated `nei_id` occurrences (82,219 instances) represent longitudinal observations of businesses across annual survey cycles rather than duplicate errors. All records were preserved.

### 8. Coordinate Treatment
- Audited coordinate columns `x` and `y`.
- Documented coordinate system as likely/inferred EPSG:3857 (WGS 84 / Pseudo-Mercator) based on coordinate bounds ($X \approx -8.87\text{M}$ to $-8.78\text{M}$, $Y \approx 5.28\text{M}$ to $5.35\text{M}$) rather than verified from official metadata.
- Identified 3 records with missing coordinates (`nei_id` 4760, 7972, 7984 from survey year 2016).
- Retained missing coordinates as `NaN` without inventing, imputing, or deleting records.

### 9. Succession Boundary & Closed/Vacant Preservation
- **Preserved Status:** Retained all 17,809 records labeled `Permanently Closed or Vacant` strictly as an operational status. Did not label or interpret them as business failure, retirement closure, or succession failure.
- **Schema Boundary Enforcement:** Zero synthetic succession variables (`owner_age`, `owner_birth_year`, `succession_plan`, `planned_exit_date`, etc.) were introduced during data cleaning.

---

## After Cleaning

### Final Dimensions
- **Final Rows:** 98,065
- **Final Columns:** 20
- **Records Removed:** **0 rows removed.** Every original business record was retained. Records with missing coordinates or secondary fields were kept to prevent survivorship and attrition bias in subsequent longitudinal and spatial analyses.

### Remaining Missing Values
- `business_po_box`: 94,275
- `business_unit`: 74,541
- `floor_size_sqft`: 55,836
- `opening_year`: 31,505
- `business_website`: 34,573
- `secondary_naics`: 19,404
- `business_postal_code`: 127
- `x`: 3
- `y`: 3

### Remaining Known Limitations
1. **Administrative Field Gaps:** Significant proportions of optional fields (`floor_size_sqft`, `opening_year`, `business_website`) are unrecorded or missing.
2. **Absence of Direct Succession Attributes:** The NEI contains no direct owner demographic, age, retirement intent, or closure reason data; proxy indicators must be developed in subsequent stages.
3. **Unprojected Points:** Exactly 3 records in 2016 lack coordinate values and cannot be mapped spatially without secondary geocoding.
4. **Inferred CRS:** Coordinate Reference System is inferred from value extents (likely EPSG:3857) and has not been verified against official Niagara Region metadata.